# Paper Explorer — Example Usage

This notebook demonstrates the public API of `paper_explorer`: crawling arXiv, embedding papers, semantic search, recommendations, topic discovery, and visualization.

Run `uv pip install -e .` from the project root before running this notebook.

In [ ]:
from paper_explorer import ArxivCrawler, Embedder, PaperStore, SearchEngine, TopicSummarizer
from paper_explorer.viz.plots import plot_embedding_space, plot_papers_per_year

## 1. Crawl papers from arXiv

In [ ]:
crawler = ArxivCrawler()
papers = crawler.search("graph neural networks", max_results=100)
len(papers), papers[0].title

## 2. Compute embeddings and persist to a local store

In [ ]:
embedder = Embedder()
embedder.embed_papers(papers)

store = PaperStore("../data/papers.json")
store.add_many(papers)
store.save()
len(store)

## 3. Semantic search

In [ ]:
engine = SearchEngine(store.all())
for paper, score in engine.search("predicting molecular properties with graph networks", top_k=5):
    print(f"{score:.3f}  {paper.title}")

## 4. Recommend papers similar to one you already like

In [ ]:
seed_id = papers[0].paper_id
for paper, score in engine.recommend(seed_id, top_k=5):
    print(f"{score:.3f}  {paper.title}")

## 5. Discover topics

In [ ]:
summarizer = TopicSummarizer(n_topics=5)
topics = summarizer.fit(store.all())
for topic in topics:
    print(topic.summary())

## 6. Visualize

In [ ]:
plot_papers_per_year(store.all(), "../data/plots/papers_per_year.png")
label_map = {p.paper_id: t.topic_id for t in topics for p in t.papers}
labeled = [p for p in store.all() if p.paper_id in label_map]
labels = [label_map[p.paper_id] for p in labeled]
plot_embedding_space(labeled, "../data/plots/embedding_space.png", labels)